# Supermarket Price Intelligence: Exploratory Data Analysis (EDA)

This notebook covers the exploratory data analysis and visualization for the Supermarket Price match pipeline. It serves as the academic report version of the Streamlit dashboard.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
We load both the initially cleaned master dataset and the strictly matched overlapping product dataset.

In [ ]:
# Load data
df_raw = pd.read_csv('../data/processed/master_cleaned_products.csv')
df_matched = pd.read_csv('../data/matched/cross_store_price_dispersion.csv')

print(f"Total Cleaned Products: {len(df_raw):,}")
print(f"Total Matched Product Pairs: {len(df_matched):,}")

## 2. Store Level Comparisons
Evaluating the sheer volume of products scraped per store.

In [ ]:
store_counts = df_raw['store'].value_counts().reset_index()
store_counts.columns = ['Store', 'Product Count']

fig = px.pie(store_counts, values='Product Count', names='Store', title='Proportion of Scraped Data by Store', hole=0.4)
fig.show()

## 3. Price Dispersion Analysis
Analyzing the discrepancies between identical matched products.

In [ ]:
fig = px.histogram(df_matched, x="price_diff_pkr", nbins=60, title="Distribution of Price Differences (PKR)")
fig.show()

### Top 10 Price Spread Anomalies
Often caused by dirty vendor API data (e.g. "Invisible Cartons" where a carton is listed as 1 piece).

In [ ]:
top_diffs = df_matched.sort_values(by='price_diff_pkr', ascending=False).head(10)
display(top_diffs[['matched_product', 'anchor_store', 'comparison_store', 'price_diff_pkr']])

fig = px.bar(top_diffs, x='price_diff_pkr', y='matched_product', orientation='h', 
             title='Top 10 Largest Price Dispersions', color='price_diff_pkr')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

## 4. Leader Dominance Index (LDI)
Which supermarket acts as the price leader most frequently across identical baskets?

In [ ]:
df_matched['price_winner'] = df_matched.apply(lambda row: 'Naheed' if row['anchor_price'] < row['comparison_price'] else row['comparison_store'], axis=1)
win_distribution = df_matched['price_winner'].value_counts().reset_index()
win_distribution.columns = ['Store', 'Lowest Price Frequency']

fig = px.bar(win_distribution, x='Store', y='Lowest Price Frequency', title='Leader Dominance Index (Lowest Price Frequency)', text_auto=True)
fig.show()

## 5. Summary Insights
- Market is highly competitive but contains distinct API-driven anomalies.
- Strict regex matching resolved most scale discrepancies (ml vs L), leaving genuine wholesale/carton mixups on Metro's side.
- Cross-store variance heavily skews towards large FMCG.